# Goodput-Analyse für OmniSphinx und native Mix-Formate

Dieses Notebook untersucht die Goodput-Effizienz verschiedener Mix-Format-Varianten.
Die betrachteten Varianten umfassen OmniSphinx in unterschiedlichen Emulationsmodi sowie native Sphinx- und PolySphinx-Formate.
Für alle Analysen setzen wir die Pfadlänge auf $r = 5$ und verwenden die in der Implementierung hinterlegte Sicherheitsparameterisierung mit $\kappa = 16$ Bytes.


## Aufgabenstellung

1. **Beta-Länge vs. Goodput**  
   * Fixierte Payload-Länge: 512 Bytes.  
   * Variiere die Beta-Länge (nur gültige Werte) und berechne Goodput sowie Paketgröße für jede Variante.  
   * Visualisiere Goodput in Abhängigkeit der Beta-Länge.

2. **Payload-Größe vs. Goodput**  
   * Fixiere die Header-Größen (Alpha, Beta, Gamma) pro Variante.  
   * Variiere die Payload-Größe von 0 bis 4096 Bytes und berechne den Goodput.  
   * Visualisiere den Goodput in Abhängigkeit der Payload-Größe.

Die Beta-Längen für die OmniSphinx-Varianten werden gemäß den bereitgestellten Formeln berechnet.
Für native Varianten orientieren wir uns an den Referenzformeln aus der Literatur bzw. der Java-Implementierung.


In [2]:
import math
from dataclasses import dataclass
from typing import Optional, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')


## Parameter und Hilfsfunktionen

Die folgenden Konstanten basieren auf der Standard-Parametrisierung aus der Codebasis:

* Sicherheitsparameter (MAC-Länge) $\kappa = 16$ Bytes.
* Elliptische Kurve **secp224r1** ⇒ unkomprimierte Punktdarstellung (Alpha) mit $1 + 2 \cdot 28 = 57$ Bytes.
* Gamma entspricht einem MAC mit Länge $\kappa$.
* Pfadlänge $r = 5$ Hops.

Zusätzlich werden Hilfsfunktionen definiert, um Beta-Längen und Goodput zu berechnen.


In [ ]:
KAPPA = 128  #Bits
ALPHA_LEN = 2*KAPPA
GAMMA_LEN = KAPPA
PATH_LENGTH = 5


def sphinx_instruction_length(kappa: int = KAPPA) -> int:
    """Länge eines OmniSphinx-Instruktionsblocks für einen Hop (inkl. Längenbyte)."""
    forward_len = 8 + 8 + kappa   # opcode + len + value 
    raw_len = (
        3*8  # concateWithByteValue, opcode + register + value
        + 3*8  # hash, opcode + inputregister + outputregister
        + 4*8  # decrypt, opcode + key + cipherRegister + outputRegister
        + forward_len  # forward
		+ 2*8 # Mixing, opcode + value
    )
    return raw_len  

def polySphinx_Relay_Instruction_length(kappa: int = KAPPA) -> int:
	load_length = 8 + 8 + kappa + 8 # opcode + Länge + Sigma + OutputRegister
	encrypt_length = 8 + 8 + 8 + 8 # opcode + SigmaRegister + PayLoadRegister +  outputRigster
	forward_len = 8 + 8 + kappa
	return load_length + encrypt_length + forward_len + 8 

def polySphinx_Exit_Instruction_length(seed: int = KAPPA,  receiver: int = KAPPA, r: int = PATH_LENGTH, log2p: int = math.ceil(math.log(PATH_LENGTH)), path: int = log2p * r) -> int: 
	load_seed = 8 + 8 + seed + 8 #Opcode + länge + seed länge + outputRegister
	load_path = 8 + 8 + path + 8 #Opcode + länge + path länge + outputRegister

	

def tau_polysphinx_post(p: int, r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
    """Beta-Länge (Post-Replikation) für OmniSphinx-PolySphinx."""
    return r * (8 + 3 * kappa) + kappa + r * math.ceil(math.log2(p)) 


def tau_polysphinx_pre(p: int, r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
    """Beta-Länge (Pre-Replikation) für native PolySphinx."""
    tau_post = tau_polysphinx_post(p, r=r, kappa=kappa)
    return (r - 1) * (8 + 3 * kappa) + 8 + p * (5 * kappa + tau_post) 


def beta_native_sphinx(r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
    """Beta-Länge für natives Sphinx (ohne Instruktions-Overhead)."""
    return r * (2 * kappa + 1)

def tau_polysphinx_omni(p: int, r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
	"Beta-Länge für OmniSphinx im PolySphinx-Emulationsmodus"

def tau_sphinx_omni(r: int = PATH_LENGTH, kappa: int = KAPPA) -> int:
	"""Beta-Länge für OmniSphinx im Sphinx-Emulationsmodus."""
	instr_len = sphinx_instruction_length(kappa)
	return r * (instr_len + kappa + 8) # Instruktionslänge + Kappa (Für MAC) + Byte für Lenge


def packet_size(payload_len: int, beta_len: int, alpha_len: int = ALPHA_LEN, gamma_len: int = GAMMA_LEN) -> int:
    """Berechne Gesamtlänge eines Pakets."""
    return alpha_len + beta_len + gamma_len + payload_len


def goodput(payload_len: int, total_len: int) -> float:
    """Goodput-Definition als Verhältnis von Nutzdaten zu übertragenen Bytes."""
    if total_len == 0:
        return 0.0
    return payload_len / total_len

	
def goodput_unicast(payload_len: int, total_len: int, p:int) -> float:
    """Goodput-Wenn der Sender um Multicast zu emulieren einfach mehrere unicast nachrichten schickt."""
    if total_len == 0:
        return 0.0
    return payload_len / (total_len * p)


@dataclass(frozen=True)
class Variant:
    name: str
    beta_min: int
    kind: str
    p: Optional[int] = None


variants: Dict[str, Variant] = {
    "omni_sphinx": Variant("OmniSphinx – Sphinx (emuliert)", tau_sphinx_omni(), "omni"),
    "omni_poly_p3": Variant("OmniSphinx – Poly (p=3)", tau_polysphinx_pre(3), "omni", 3),
    "omni_poly_p5": Variant("OmniSphinx – Poly (p=5)", tau_polysphinx_pre(5), "omni", 5),
    "omni_poly_p10": Variant("OmniSphinx – Poly (p=10)", tau_polysphinx_pre(10), "omni", 10),
    "sphinx_native": Variant("Sphinx (nativ)", beta_native_sphinx(), "native"),
    "poly_native_p3": Variant("PolySphinx (nativ, p=3)", tau_polysphinx_pre(3), "native", 3),
    "poly_native_p5": Variant("PolySphinx (nativ, p=5)", tau_polysphinx_pre(5), "native", 5),
    "poly_native_p10": Variant("PolySphinx (nativ, p=10)", tau_polysphinx_pre(10), "native", 10),
}

summary_df = pd.DataFrame(
    {
        "Variante": [v.name for v in variants.values()],
        "βₘᵢₙ [B]": [v.beta_min for v in variants.values()],
        "Replikationsfaktor": [v.p if v.p is not None else "–" for v in variants.values()],
    }
)
summary_df


,Variante,βₘᵢₙ [B],Replikationsfaktor
0,OmniSphinx – Sphinx (emuliert),235,–
1,OmniSphinx – Poly (p=3),1215,3
2,OmniSphinx – Poly (p=5),1897,5
3,OmniSphinx – Poly (p=10),3602,10
4,Sphinx (nativ),165,–
5,"PolySphinx (nativ, p=3)",1215,3
6,"PolySphinx (nativ, p=5)",1897,5
7,"PolySphinx (nativ, p=10)",3602,10
